In [5]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(r'C:\Users\User\Downloads\Data Analyst Project\Upi_Fraud_Analysis\Data\synthetic_indian_upi_fraud_data.csv')


# ============================================================
# 2. BASIC CLEANING
# ============================================================

# Remove duplicate rows
df = df.drop_duplicates()

# Convert timestamp to datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

# Convert numeric columns
numeric_columns = [
    "Amount",
    "Latitude",
    "Longitude",
    "AvgTransactionAmount",
    "TransactionFrequency",
    "FailedAttempts",
    "FraudFlag"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ============================================================
# 3. HANDLE MISSING VALUES
# ============================================================

# Numeric columns → median
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

# Text columns → Unknown
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].fillna("Unknown")


# ============================================================
# 4. CREATE DATE/TIME FEATURES
# ============================================================

df["Date"] = df["Timestamp"].dt.date

df["Year"] = df["Timestamp"].dt.year

df["Month"] = df["Timestamp"].dt.month

df["MonthName"] = df["Timestamp"].dt.strftime("%B")

df["Day"] = df["Timestamp"].dt.day

df["DayOfWeek"] = df["Timestamp"].dt.day_name()

df["Hour"] = df["Timestamp"].dt.hour


# ============================================================
# 5. CREATE TRANSACTION AMOUNT BANDS
# ============================================================

df["AmountBand"] = pd.cut(
    df["Amount"],
    bins=[0, 1000, 2500, 5000, 7500, 10000, np.inf],
    labels=[
        "0-1K",
        "1K-2.5K",
        "2.5K-5K",
        "5K-7.5K",
        "7.5K-10K",
        "10K+"
    ],
    include_lowest=True
)


# ============================================================
# 6. CREATE FRAUD STATUS
# ============================================================

df["FraudStatus"] = df["FraudFlag"].map({
    0: "Not Fraud",
    1: "Fraud"
})


# ============================================================
# 7. CREATE RISK SCORE
# ============================================================

df["RiskScore"] = 0

# New device
df.loc[df["NewDevice"] == 1, "RiskScore"] += 25

# Unusual location
df.loc[df["UnusualLocation"] == 1, "RiskScore"] += 25

# Unusual amount
df.loc[df["UnusualAmount"] == 1, "RiskScore"] += 25

# Multiple failed attempts
df.loc[df["FailedAttempts"] >= 3, "RiskScore"] += 25


# ============================================================
# 8. CREATE RISK LEVEL
# ============================================================

df["RiskLevel"] = pd.cut(
    df["RiskScore"],
    bins=[-1, 25, 50, 75, 100],
    labels=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]
)


# ============================================================
# 9. CREATE HIGH VALUE TRANSACTION FLAG
# ============================================================

df["HighValueTransaction"] = np.where(
    df["Amount"] >= 5000,
    "High Value",
    "Normal"
)


# ============================================================
# 10. CREATE FAILED ATTEMPT CATEGORY
# ============================================================

df["FailedAttemptCategory"] = pd.cut(
    df["FailedAttempts"],
    bins=[-1, 0, 2, 5, np.inf],
    labels=[
        "No Failed Attempts",
        "1-2 Attempts",
        "3-5 Attempts",
        "6+ Attempts"
    ]
)


# ============================================================
# 11. REORDER IMPORTANT COLUMNS
# ============================================================

important_columns = [
    "TransactionID",
    "UserID",
    "Timestamp",
    "Date",
    "Year",
    "Month",
    "MonthName",
    "Day",
    "DayOfWeek",
    "Hour",
    "Amount",
    "AmountBand",
    "HighValueTransaction",
    "MerchantCategory",
    "TransactionType",
    "BankName",
    "DeviceID",
    "IPAddress",
    "Latitude",
    "Longitude",
    "AvgTransactionAmount",
    "TransactionFrequency",
    "UnusualLocation",
    "UnusualAmount",
    "NewDevice",
    "FailedAttempts",
    "FailedAttemptCategory",
    "FraudFlag",
    "FraudStatus",
    "RiskScore",
    "RiskLevel",
    "PhoneNumber"
]

df = df[important_columns]


# ============================================================
# 12. FINAL DATA QUALITY CHECK
# ============================================================

print("=" * 60)
print("FINAL DATASET INFORMATION")
print("=" * 60)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nFraud Distribution:")
print(df["FraudStatus"].value_counts())

print("\nRisk Level Distribution:")
print(df["RiskLevel"].value_counts())

print("\nRisk Score Distribution:")
print(df["RiskScore"].value_counts().sort_index())




FINAL DATASET INFORMATION
Rows: 10000
Columns: 32

Missing Values:
20000

Duplicate Rows:
0

Fraud Distribution:
Series([], Name: count, dtype: int64)

Risk Level Distribution:
RiskLevel
Medium      3770
Low         3487
High        2244
Critical     499
Name: count, dtype: int64

Risk Score Distribution:
RiskScore
0       750
25     2737
50     3770
75     2244
100     499
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_8476\1910712717.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns
